In [1]:
!pip install -U anthropic pandas tqdm python-dotenv

     |████████████████████████████████| 763 kB 3.8 MB/s            
  Attempting uninstall: anthropic
    Found existing installation: anthropic 0.97.0
    Uninstalling anthropic-0.97.0:
      Successfully uninstalled anthropic-0.97.0


In [11]:
from __future__ import annotations

import os
import re
import json
import time
import uuid
import hashlib
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
import anthropic

load_dotenv()

# assert os.getenv("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY. Put it in your environment or .env file."

# client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic(api_key="")

# -----------------------------
# Experiment configuration
# -----------------------------
PROVIDER = "anthropic"
MODEL_NAME = "claude-sonnet-4-6"
TEMPERATURE = 1.0

# Anthropic does not have OpenAI-style `reasoning_effort` / `verbosity`.
# We do NOT enable extended thinking for this constrained creative-generation task.
ANTHROPIC_THINKING = None

# Prompt caching is off by default because our prompts are likely below the
# 1,024-token minimum cacheable prompt length for Claude Sonnet 4.6.
ANTHROPIC_ENABLE_PROMPT_CACHING = False
ANTHROPIC_CACHE_CONTROL = {"type": "ephemeral"}  # only used if enabled

N_BASE_AGENTS = 150
N_DYADS = 75
N_TRIADS = 50

MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 60,
    "aut": 120,
    "story": 700,
}

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = Path("ai_data") / "deflect_creativity" / PROVIDER / f"model_{MODEL_NAME}" / f"run_{RUN_ID}"

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "round1_plans": DATA_ROOT / "01_round1" / "plans",
    "round1_batch_inputs": DATA_ROOT / "01_round1" / "batch_inputs",
    "round1_manifests": DATA_ROOT / "01_round1" / "manifests",
    "round1_raw_outputs": DATA_ROOT / "01_round1" / "raw_outputs",
    "round1_parsed": DATA_ROOT / "01_round1" / "parsed",
    "round2_plans": DATA_ROOT / "02_round2" / "plans",
    "round2_batch_inputs": DATA_ROOT / "02_round2" / "batch_inputs",
    "round2_manifests": DATA_ROOT / "02_round2" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "02_round2" / "raw_outputs",
    "round2_parsed": DATA_ROOT / "02_round2" / "parsed",
    "compiled": DATA_ROOT / "03_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

Run directory:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3


In [3]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text


def to_jsonable(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    return obj

In [4]:
run_config = {
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "temperature": TEMPERATURE,
    "anthropic_thinking": ANTHROPIC_THINKING,
    "anthropic_enable_prompt_caching": ANTHROPIC_ENABLE_PROMPT_CACHING,
    "anthropic_cache_control": ANTHROPIC_CACHE_CONTROL if ANTHROPIC_ENABLE_PROMPT_CACHING else None,
    "n_base_agents": N_BASE_AGENTS,
    "n_dyads": N_DYADS,
    "n_triads": N_TRIADS,
    "max_output_tokens_by_family": MAX_OUTPUT_TOKENS_BY_FAMILY,
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/00_metadata/experiment_config__20260517_205551__0fab9ccf.json')

In [5]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_smartphone",
        "task_family": "slogan",
        "task_label": "Smartphone slogan",
        "task_prompt_key": "smartphone",
    },
    {
        "task_id": "slogan_soda",
        "task_family": "slogan",
        "task_label": "Soda slogan",
        "task_prompt_key": "soda",
    },
    {
        "task_id": "aut_shoe",
        "task_family": "aut",
        "task_label": "AUT shoe",
        "task_prompt_key": "shoe",
        "object": "shoe",
        "common_use": "used as footwear",
    },
    {
        "task_id": "aut_button",
        "task_family": "aut",
        "task_label": "AUT button",
        "task_prompt_key": "button",
        "object": "button",
        "common_use": "used to fasten things",
    },
    {
        "task_id": "story_jungle",
        "task_family": "story",
        "task_label": "Jungle adventure story",
        "task_prompt_key": "jungle",
    },
    {
        "task_id": "story_parachute",
        "task_family": "story",
        "task_label": "Parachute story",
        "task_prompt_key": "parachute",
    },
]

STRATEGIES = ["vanilla", "diverge"]
CONDITIONS = ["base", "dyad", "triad"]

pd.DataFrame(TASK_SETTINGS)

,task_id,task_family,task_label,task_prompt_key,object,common_use
0,slogan_smartphone,slogan,Smartphone slogan,smartphone,NaN,NaN
1,slogan_soda,slogan,Soda slogan,soda,NaN,NaN
2,aut_shoe,aut,AUT shoe,shoe,shoe,used as footwear
3,aut_button,aut,AUT button,button,button,used to fasten things
4,story_jungle,story,Jungle adventure story,jungle,NaN,NaN
5,story_parachute,story,Parachute story,parachute,NaN,NaN


In [6]:
SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_smartphone":
        return (
            "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\n"
            "Generate exactly one marketing slogan for this brand-new smartphone.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the smartphone.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_soda":
        return (
            "You are part of the marketing team at a beverage company preparing to launch a new soda.\n\n"
            "Generate exactly one marketing slogan for this brand-new soda.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the soda.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_shoe", "aut_button"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_jungle":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story about an adventure in the jungle.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_parachute":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story based on this prompt:\n"
            "The parachute isn’t opening up.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def build_round1_prompt(task: dict, strategy: str) -> str:
    return base_task_prompt(task) + "\n\n" + strategy_block(strategy)


def build_round2_context(
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    if condition == "base":
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
        )
    elif condition == "dyad":
        assert len(peer_round1_texts) == 1
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous response from another agent in the same first round:\n'
            f'"{peer_round1_texts[0]}"\n\n'
        )
    elif condition == "triad":
        assert len(peer_round1_texts) == 2
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous responses from two other agents in the same first round:\n'
            f'1. "{peer_round1_texts[0]}"\n'
            f'2. "{peer_round1_texts[1]}"\n\n'
        )
    else:
        raise ValueError(f"Unknown condition: {condition}")

    if strategy == "vanilla":
        return context + "Now generate one new response for the same task."

    if strategy == "diverge":
        return (
            context
            + "Now generate one new response for the same task. "
              "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def build_round2_prompt(
    task: dict,
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + build_round2_context(
            strategy=strategy,
            condition=condition,
            self_round1=self_round1,
            peer_round1_texts=peer_round1_texts,
        )
    )

In [7]:
def build_agent_roster() -> pd.DataFrame:
    rows = []

    for i in range(1, N_BASE_AGENTS + 1):
        rows.append({
            "condition": "base",
            "group_id": f"base_{i:03d}",
            "group_size": 1,
            "agent_index": 1,
            "agent_id": f"base_{i:03d}__a1",
        })

    for g in range(1, N_DYADS + 1):
        for a in [1, 2]:
            rows.append({
                "condition": "dyad",
                "group_id": f"dyad_{g:03d}",
                "group_size": 2,
                "agent_index": a,
                "agent_id": f"dyad_{g:03d}__a{a}",
            })

    for g in range(1, N_TRIADS + 1):
        for a in [1, 2, 3]:
            rows.append({
                "condition": "triad",
                "group_id": f"triad_{g:03d}",
                "group_size": 3,
                "agent_index": a,
                "agent_id": f"triad_{g:03d}__a{a}",
            })

    return pd.DataFrame(rows)


agents_df = build_agent_roster()

display(agents_df.groupby("condition").agg(
    n_agents=("agent_id", "count"),
    n_groups=("group_id", "nunique"),
    group_size=("group_size", "first"),
))

agents_df.head()

,n_agents,n_groups,group_size
condition,,,
base,150,150,1
dyad,150,75,2
triad,150,50,3


,condition,group_id,group_size,agent_index,agent_id
0,base,base_001,1,1,base_001__a1
1,base,base_002,1,1,base_002__a1
2,base,base_003,1,1,base_003__a1
3,base,base_004,1,1,base_004__a1
4,base,base_005,1,1,base_005__a1


In [8]:
def build_round1_plan() -> pd.DataFrame:
    rows = []

    for task in TASK_SETTINGS:
        for strategy in STRATEGIES:
            for _, agent in agents_df.iterrows():
                user_prompt = build_round1_prompt(task, strategy)
                request_basis = {
                    "provider": PROVIDER,
                    "model": MODEL_NAME,
                    "round": 1,
                    "task_id": task["task_id"],
                    "task_family": task["task_family"],
                    "strategy": strategy,
                    "condition": agent["condition"],
                    "group_id": agent["group_id"],
                    "agent_id": agent["agent_id"],
                    "agent_index": int(agent["agent_index"]),
                }
                request_key = "r1__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                rows.append({
                    **request_basis,
                    "request_key": request_key,
                    "system_instructions": SYSTEM_INSTRUCTIONS,
                    "user_prompt": user_prompt,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                    "created_at_utc": now_iso(),
                })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round1_plan_df = build_round1_plan()

print(round1_plan_df.shape)
display(round1_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round1_plan_df.head()

(5400, 16)


,task_id,strategy,condition,n
0,aut_button,diverge,base,150
1,aut_button,diverge,dyad,150
2,aut_button,diverge,triad,150
3,aut_button,vanilla,base,150
4,aut_button,vanilla,dyad,150
5,aut_button,vanilla,triad,150
6,aut_shoe,diverge,base,150
7,aut_shoe,diverge,dyad,150
8,aut_shoe,diverge,triad,150
9,aut_shoe,vanilla,base,150


,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,request_key,system_instructions,user_prompt,temperature,max_output_tokens,created_at_utc
0,anthropic,claude-sonnet-4-6,1,slogan_smartphone,slogan,vanilla,base,base_001,base_001__a1,1,r1__e5842e2e58a5123399f4145e,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:56:02.216549+00:00
1,anthropic,claude-sonnet-4-6,1,slogan_smartphone,slogan,vanilla,base,base_002,base_002__a1,1,r1__b1bbef3536edc1017ac626ca,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:56:02.216705+00:00
2,anthropic,claude-sonnet-4-6,1,slogan_smartphone,slogan,vanilla,base,base_003,base_003__a1,1,r1__ee38f9b4ad640f85d2496a9b,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:56:02.216925+00:00
3,anthropic,claude-sonnet-4-6,1,slogan_smartphone,slogan,vanilla,base,base_004,base_004__a1,1,r1__f1c08bd8c9fbb99acd100ed9,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:56:02.217528+00:00
4,anthropic,claude-sonnet-4-6,1,slogan_smartphone,slogan,vanilla,base,base_005,base_005__a1,1,r1__82321f2a757c067458735baf,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:56:02.217734+00:00


In [9]:
def make_anthropic_message_params(row: pd.Series) -> dict:
    params = {
        "model": MODEL_NAME,
        "max_tokens": int(row["max_output_tokens"]),
        "temperature": float(row["temperature"]),
        "system": row["system_instructions"],
        "messages": [
            {
                "role": "user",
                "content": row["user_prompt"],
            }
        ],
    }

    if ANTHROPIC_THINKING is not None:
        params["thinking"] = ANTHROPIC_THINKING

    if ANTHROPIC_ENABLE_PROMPT_CACHING:
        # Automatic caching. Likely no-op for this project unless prompts exceed the minimum cacheable length.
        params["cache_control"] = ANTHROPIC_CACHE_CONTROL

    return params


def make_anthropic_batch_request_files(
    plan_df: pd.DataFrame,
    round_name: str,
    batch_input_dir: Path,
) -> tuple[list[dict], Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{round_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = batch_input_dir.parent / "plans" / f"{stem}__plan.csv"
    jsonl_path = batch_input_dir / f"{stem}__local_batch_requests.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    plan_df.to_csv(plan_path, index=False)

    requests = []
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request = {
                "custom_id": row["request_key"],
                "params": make_anthropic_message_params(row),
            }
            requests.append(request)
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"Wrote plan:          {plan_path}")
    print(f"Wrote local JSONL:   {jsonl_path}")
    print(f"Batch request count: {len(requests):,}")

    return requests, jsonl_path, plan_path


round1_requests, round1_jsonl_path, round1_plan_path = make_anthropic_batch_request_files(
    plan_df=round1_plan_df,
    round_name="round1",
    batch_input_dir=DIRS["round1_batch_inputs"],
)

round1_jsonl_path, round1_plan_path

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/01_round1/plans/round1__anthropic__claude-sonnet-4-6__20260517_205605__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/01_round1/batch_inputs/round1__anthropic__claude-sonnet-4-6__20260517_205605__local_batch_requests.jsonl
Batch request count: 5,400


(PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/01_round1/batch_inputs/round1__anthropic__claude-sonnet-4-6__20260517_205605__local_batch_requests.jsonl'),
 PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/01_round1/plans/round1__anthropic__claude-sonnet-4-6__20260517_205605__plan.csv'))

In [12]:
def submit_anthropic_batch(
    requests: list[dict],
    round_name: str,
    plan_path: Path,
    local_jsonl_path: Path,
    manifest_dir: Path,
) -> dict:
    message_batch = client.messages.batches.create(requests=requests)

    batch_dump = to_jsonable(message_batch)

    batch_info = {
        "run_id": RUN_ID,
        "round": round_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "message_batch": batch_dump,
        "batch_id": batch_dump.get("id"),
        "processing_status_at_submission": batch_dump.get("processing_status"),
        "submitted_at_utc": now_iso(),
        "local_jsonl_path": str(local_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = manifest_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{batch_info['batch_id']}.json"
    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted Anthropic Message Batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


round1_batch_info = submit_anthropic_batch(
    requests=round1_requests,
    round_name="round1",
    plan_path=round1_plan_path,
    local_jsonl_path=round1_jsonl_path,
    manifest_dir=DIRS["round1_manifests"],
)

round1_batch_info

Submitted Anthropic Message Batch:
{
  "run_id": "20260517_205619__040c8ea3",
  "round": "round1",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_014jXGJ38KdqcDcFqm1ESNfc",
    "archived_at": null,
    "cancel_initiated_at": null,
    "created_at": "2026-05-18T00:56:28.154851Z",
    "ended_at": null,
    "expires_at": "2026-05-19T00:56:28.154851Z",
    "processing_status": "in_progress",
    "request_counts": {
      "canceled": 0,
      "errored": 0,
      "expired": 0,
      "processing": 5400,
      "succeeded": 0
    },
    "results_url": null,
    "type": "message_batch"
  },
  "batch_id": "msgbatch_014jXGJ38KdqcDcFqm1ESNfc",
  "processing_status_at_submission": "in_progress",
  "submitted_at_utc": "2026-05-18T00:56:28.596907+00:00",
  "local_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/01_round1/batch_inputs/round1__anthropic__claude-sonnet-4-6__20260517_205605__local_

{'run_id': '20260517_205619__040c8ea3',
 'round': 'round1',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_014jXGJ38KdqcDcFqm1ESNfc',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-18T00:56:28.154851Z',
  'ended_at': None,
  'expires_at': '2026-05-19T00:56:28.154851Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 5400,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_014jXGJ38KdqcDcFqm1ESNfc',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-18T00:56:28.596907+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205551__0fab9ccf/01_round1/batch_inputs/round1__anthropic__claude-sonnet-4-6__20260517_205605__local_batch_requests.jsonl',
 'plan_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4

In [17]:
def check_anthropic_batch(batch_id: str) -> dict:
    message_batch = client.messages.batches.retrieve(batch_id)
    info = to_jsonable(message_batch)
    print(json.dumps(info, indent=2))
    return info


round1_status = check_anthropic_batch(round1_batch_info["batch_id"])
round1_status

{
  "id": "msgbatch_014jXGJ38KdqcDcFqm1ESNfc",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-18T00:56:28.154851Z",
  "ended_at": "2026-05-18T01:04:43.401896Z",
  "expires_at": "2026-05-19T00:56:28.154851Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 5400
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_014jXGJ38KdqcDcFqm1ESNfc/results",
  "type": "message_batch"
}


{'id': 'msgbatch_014jXGJ38KdqcDcFqm1ESNfc',
 'archived_at': None,
 'cancel_initiated_at': None,
 'created_at': '2026-05-18T00:56:28.154851Z',
 'ended_at': '2026-05-18T01:04:43.401896Z',
 'expires_at': '2026-05-19T00:56:28.154851Z',
 'processing_status': 'ended',
 'request_counts': {'canceled': 0,
  'errored': 0,
  'expired': 0,
  'processing': 0,
  'succeeded': 5400},
 'results_url': 'https://api.anthropic.com/v1/messages/batches/msgbatch_014jXGJ38KdqcDcFqm1ESNfc/results',
 'type': 'message_batch'}

In [ ]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_.../01_round1/manifests/round1__anthropic__claude-sonnet-4-6__batch_manifest__msgbatch_....json")
# round1_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round1_batch_info["data_root"])
# round1_batch_info

In [18]:
def download_anthropic_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    round_name: str,
) -> Optional[Path]:
    message_batch = client.messages.batches.retrieve(batch_id)
    batch_dump = to_jsonable(message_batch)

    if batch_dump.get("processing_status") != "ended":
        print(f"Batch is not ended yet. Current status: {batch_dump.get('processing_status')}")
        return None

    output_path = raw_output_dir / f"{round_name}__{batch_id}__results.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output file: {output_path}")

    n = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for result in client.messages.batches.results(batch_id):
            result_dict = to_jsonable(result)
            f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")
            n += 1

    print(f"Downloaded/streamed {n:,} results to: {output_path}")
    return output_path


round1_output_path = download_anthropic_batch_results(
    batch_id=round1_batch_info["batch_id"],
    raw_output_dir=DIRS["round1_raw_outputs"],
    round_name="round1",
)

round1_output_path

Downloaded/streamed 5,400 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/01_round1/raw_outputs/round1__msgbatch_014jXGJ38KdqcDcFqm1ESNfc__results.jsonl


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/01_round1/raw_outputs/round1__msgbatch_014jXGJ38KdqcDcFqm1ESNfc__results.jsonl')

In [19]:
def extract_text_from_anthropic_message(message: dict) -> str:
    if not isinstance(message, dict):
        return ""

    texts = []
    for block in message.get("content", []) or []:
        if isinstance(block, dict) and block.get("type") == "text":
            texts.append(block.get("text", ""))

    return "\n".join(texts).strip()


def parse_anthropic_batch_output_to_standard_files(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    round_name: str,
    batch_id: str,
) -> dict:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing parsed JSONL: {parsed_jsonl_path}")

    parsed_records = []
    n_success = 0
    n_empty = 0
    n_error = 0
    n_canceled = 0
    n_expired = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        result = rec.get("result", {})
        result_type = result.get("type")

        if result_type == "succeeded":
            message = result.get("message", {})
            text = clean_model_text(extract_text_from_anthropic_message(message))
            usage = message.get("usage")

            status = "success" if text else "empty_text"
            n_success += int(status == "success")
            n_empty += int(status == "empty_text")

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": message.get("id"),
                "stop_reason": message.get("stop_reason"),
                "usage": usage,
                "error": None if text else "No text extracted from Anthropic message.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
                "raw_result_type": result_type,
            }

        else:
            if result_type == "errored":
                n_error += 1
            elif result_type == "canceled":
                n_canceled += 1
            elif result_type == "expired":
                n_expired += 1
            else:
                n_error += 1

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": result_type or "unknown_error",
                "text": None,
                "provider_response_id": None,
                "stop_reason": None,
                "usage": None,
                "error": result.get("error"),
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
                "raw_result_type": result_type,
            }

        parsed_records.append(record)
        append_jsonl(parsed_jsonl_path, record)

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "round": round_name,
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": n_success,
        "n_empty_text": n_empty,
        "n_error": n_error,
        "n_canceled": n_canceled,
        "n_expired": n_expired,
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(summary, indent=2))
    return summary


round1_parse_summary = parse_anthropic_batch_output_to_standard_files(
    batch_output_path=round1_output_path,
    plan_path=Path(round1_batch_info["plan_path"]),
    parsed_dir=DIRS["round1_parsed"],
    round_name="round1",
    batch_id=round1_batch_info["batch_id"],
)

round1_df = pd.read_pickle(round1_parse_summary["parsed_pkl_path"])
print(round1_df.shape)
display(round1_df["status"].value_counts(dropna=False))
round1_df.head()

{
  "round": "round1",
  "batch_id": "msgbatch_014jXGJ38KdqcDcFqm1ESNfc",
  "n_records": 5400,
  "n_success": 5400,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/01_round1/parsed/round1__anthropic__claude-sonnet-4-6__msgbatch_014jXGJ38KdqcDcFqm1ESNfc__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/01_round1/parsed/round1__anthropic__claude-sonnet-4-6__msgbatch_014jXGJ38KdqcDcFqm1ESNfc__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/01_round1/parsed/round1__anthropic__claude-sonnet-4-6__msgbatch_014jXGJ38KdqcDcFqm1ESNfc__parsed.pkl"
}
(5400, 27)


status
success    5400
Name: count, dtype: int64

,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,status,text,provider_response_id,stop_reason,usage,error,batch_custom_id,batch_output_file,batch_id,raw_result_type
0,anthropic,claude-sonnet-4-6,1,story_parachute,story,vanilla,dyad,dyad_074,dyad_074__a1,1,...,success,Maya's fingers clawed at the tangled cords as ...,msg_01MFTTdGE5UDXaXGTcEgkryf,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__bdae884b3eae5a88a07f9d6e,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_014jXGJ38KdqcDcFqm1ESNfc,succeeded
1,anthropic,claude-sonnet-4-6,1,aut_button,aut,vanilla,triad,triad_048,triad_048__a3,3,...,success,Using a button as a tiny canvas for miniature ...,msg_01Bd8wUi5xSxLJgbiFeyrdhX,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__0f9da072f1a89681d8922e6c,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_014jXGJ38KdqcDcFqm1ESNfc,succeeded
2,anthropic,claude-sonnet-4-6,1,slogan_soda,slogan,diverge,dyad,dyad_059,dyad_059__a2,2,...,success,Fizz louder than your Monday morning.,msg_01FQQb97XH6mrtRkGTbZiqqT,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__f1d0b76030561a5b13866698,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_014jXGJ38KdqcDcFqm1ESNfc,succeeded
3,anthropic,claude-sonnet-4-6,1,aut_button,aut,diverge,base,base_033,base_033__a1,1,...,success,Glue a cluster of mismatched buttons onto a ca...,msg_017RYxfdQFdFJ9QokeD6y4y2,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__0167997fdb7adc2d910eaee5,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_014jXGJ38KdqcDcFqm1ESNfc,succeeded
4,anthropic,claude-sonnet-4-6,1,aut_shoe,aut,vanilla,dyad,dyad_043,dyad_043__a1,1,...,success,A shoe can be used as a planter for small succ...,msg_012PA84RgKwxLz9aNgjLHh1o,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__7a2c42c8aa78c3e855ec9da2,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_014jXGJ38KdqcDcFqm1ESNfc,succeeded


In [20]:
expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)
actual_round1 = len(round1_df)

print("Expected Round 1 rows:", expected_round1)
print("Actual Round 1 rows:  ", actual_round1)

display(round1_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round1 != expected_round1:
    print("WARNING: Row count mismatch. Inspect before proceeding.")

if (round1_df["status"] != "success").any():
    print("WARNING: Some Round 1 calls failed or returned empty text. Inspect before proceeding to Round 2.")
    display(round1_df[round1_df["status"] != "success"].head(20))
else:
    print("Round 1 looks complete.")

Expected Round 1 rows: 5400
Actual Round 1 rows:   5400


,task_id,strategy,condition,status,n
0,aut_button,diverge,base,success,150
1,aut_button,diverge,dyad,success,150
2,aut_button,diverge,triad,success,150
3,aut_button,vanilla,base,success,150
4,aut_button,vanilla,dyad,success,150
5,aut_button,vanilla,triad,success,150
6,aut_shoe,diverge,base,success,150
7,aut_shoe,diverge,dyad,success,150
8,aut_shoe,diverge,triad,success,150
9,aut_shoe,vanilla,base,success,150


Round 1 looks complete.


In [21]:
def get_task_by_id(task_id: str) -> dict:
    for t in TASK_SETTINGS:
        if t["task_id"] == task_id:
            return t
    raise KeyError(task_id)


def build_round2_plan(round1_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    r1_success = round1_df[round1_df["status"] == "success"].copy()

    required_cols = ["task_id", "strategy", "condition", "group_id", "agent_id"]
    if r1_success.duplicated(required_cols).any():
        dupes = r1_success[r1_success.duplicated(required_cols, keep=False)].sort_values(required_cols)
        raise ValueError(f"Duplicate Round 1 successful records:\n{dupes[required_cols + ['text']].head()}")

    for (task_id, strategy, condition, group_id), group in r1_success.groupby(
        ["task_id", "strategy", "condition", "group_id"],
        sort=True,
    ):
        task = get_task_by_id(task_id)
        group = group.sort_values("agent_index").copy()

        expected_group_size = {"base": 1, "dyad": 2, "triad": 3}[condition]
        if len(group) != expected_group_size:
            raise ValueError(
                f"Group size mismatch for {(task_id, strategy, condition, group_id)}: "
                f"expected {expected_group_size}, got {len(group)}"
            )

        for _, ego in group.iterrows():
            peer_rows = group[group["agent_id"] != ego["agent_id"]].sort_values("agent_index")
            peer_texts = peer_rows["text"].tolist()
            peer_agent_ids = peer_rows["agent_id"].tolist()

            user_prompt = build_round2_prompt(
                task=task,
                strategy=strategy,
                condition=condition,
                self_round1=ego["text"],
                peer_round1_texts=peer_texts,
            )

            request_basis = {
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "round": 2,
                "task_id": task_id,
                "task_family": task["task_family"],
                "strategy": strategy,
                "condition": condition,
                "group_id": group_id,
                "agent_id": ego["agent_id"],
                "agent_index": int(ego["agent_index"]),
                "self_round1_request_key": ego["request_key"],
                "peer_round1_agent_ids": "|".join(peer_agent_ids),
            }
            request_key = "r2__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

            rows.append({
                **request_basis,
                "request_key": request_key,
                "system_instructions": SYSTEM_INSTRUCTIONS,
                "user_prompt": user_prompt,
                "temperature": TEMPERATURE,
                "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                "self_round1_text": ego["text"],
                "peer_round1_texts_json": json.dumps(peer_texts, ensure_ascii=False),
                "created_at_utc": now_iso(),
            })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round2_plan_df = build_round2_plan(round1_df)

print(round2_plan_df.shape)
display(round2_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round2_plan_df.head()

(5400, 20)


,task_id,strategy,condition,n
0,aut_button,diverge,base,150
1,aut_button,diverge,dyad,150
2,aut_button,diverge,triad,150
3,aut_button,vanilla,base,150
4,aut_button,vanilla,dyad,150
5,aut_button,vanilla,triad,150
6,aut_shoe,diverge,base,150
7,aut_shoe,diverge,dyad,150
8,aut_shoe,diverge,triad,150
9,aut_shoe,vanilla,base,150


,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,self_round1_request_key,peer_round1_agent_ids,request_key,system_instructions,user_prompt,temperature,max_output_tokens,self_round1_text,peer_round1_texts_json,created_at_utc
0,anthropic,claude-sonnet-4-6,2,aut_button,aut,diverge,base,base_001,base_001__a1,1,r1__4718bbc27c8658f239ac6345,,r2__a876b11238b185a7d5676f33,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Using a button as a tiny canvas for miniature ...,[],2026-05-18T01:09:23.608786+00:00
1,anthropic,claude-sonnet-4-6,2,aut_button,aut,diverge,base,base_002,base_002__a1,1,r1__717737e987a9810b894d47bb,,r2__0c51631c832e7ea01f553e0d,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Using a button as a tiny palette for mixing sm...,[],2026-05-18T01:09:23.609337+00:00
2,anthropic,claude-sonnet-4-6,2,aut_button,aut,diverge,base,base_003,base_003__a1,1,r1__a6575db26dffd03b41b45bd1,,r2__6c58966bd746dab0f0a5896f,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Glue a collection of mismatched buttons onto a...,[],2026-05-18T01:09:23.610126+00:00
3,anthropic,claude-sonnet-4-6,2,aut_button,aut,diverge,base,base_004,base_004__a1,1,r1__58b3a313f56345a7f041fa17,,r2__700576ba9d04fa4b88cf3432,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,"Use a button as a tiny painter's palette, plac...",[],2026-05-18T01:09:23.610808+00:00
4,anthropic,claude-sonnet-4-6,2,aut_button,aut,diverge,base,base_005,base_005__a1,1,r1__3d611088b91f87e5302886de,,r2__8be51dc1d8c0a7474f675ed7,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Use a large decorative button as a miniature p...,[],2026-05-18T01:09:23.611317+00:00


In [22]:
round2_requests, round2_jsonl_path, round2_plan_path = make_anthropic_batch_request_files(
    plan_df=round2_plan_df,
    round_name="round2",
    batch_input_dir=DIRS["round2_batch_inputs"],
)

round2_jsonl_path, round2_plan_path

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/plans/round2__anthropic__claude-sonnet-4-6__20260517_210929__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/batch_inputs/round2__anthropic__claude-sonnet-4-6__20260517_210929__local_batch_requests.jsonl
Batch request count: 5,400


(PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/batch_inputs/round2__anthropic__claude-sonnet-4-6__20260517_210929__local_batch_requests.jsonl'),
 PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/plans/round2__anthropic__claude-sonnet-4-6__20260517_210929__plan.csv'))

In [23]:
round2_batch_info = submit_anthropic_batch(
    requests=round2_requests,
    round_name="round2",
    plan_path=round2_plan_path,
    local_jsonl_path=round2_jsonl_path,
    manifest_dir=DIRS["round2_manifests"],
)

round2_batch_info

Submitted Anthropic Message Batch:
{
  "run_id": "20260517_205619__040c8ea3",
  "round": "round2",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ",
    "archived_at": null,
    "cancel_initiated_at": null,
    "created_at": "2026-05-18T01:09:34.328513Z",
    "ended_at": null,
    "expires_at": "2026-05-19T01:09:34.328513Z",
    "processing_status": "in_progress",
    "request_counts": {
      "canceled": 0,
      "errored": 0,
      "expired": 0,
      "processing": 5400,
      "succeeded": 0
    },
    "results_url": null,
    "type": "message_batch"
  },
  "batch_id": "msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ",
  "processing_status_at_submission": "in_progress",
  "submitted_at_utc": "2026-05-18T01:09:34.739686+00:00",
  "local_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/batch_inputs/round2__anthropic__claude-sonnet-4-6__20260517_210929__local_

{'run_id': '20260517_205619__040c8ea3',
 'round': 'round2',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-18T01:09:34.328513Z',
  'ended_at': None,
  'expires_at': '2026-05-19T01:09:34.328513Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 5400,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-18T01:09:34.739686+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/batch_inputs/round2__anthropic__claude-sonnet-4-6__20260517_210929__local_batch_requests.jsonl',
 'plan_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4

In [29]:
round2_status = check_anthropic_batch(round2_batch_info["batch_id"])
round2_status

{
  "id": "msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-18T01:09:34.328513Z",
  "ended_at": "2026-05-18T01:17:20.101594Z",
  "expires_at": "2026-05-19T01:09:34.328513Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 5400
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ/results",
  "type": "message_batch"
}


{'id': 'msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ',
 'archived_at': None,
 'cancel_initiated_at': None,
 'created_at': '2026-05-18T01:09:34.328513Z',
 'ended_at': '2026-05-18T01:17:20.101594Z',
 'expires_at': '2026-05-19T01:09:34.328513Z',
 'processing_status': 'ended',
 'request_counts': {'canceled': 0,
  'errored': 0,
  'expired': 0,
  'processing': 0,
  'succeeded': 5400},
 'results_url': 'https://api.anthropic.com/v1/messages/batches/msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ/results',
 'type': 'message_batch'}

In [30]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_.../02_round2/manifests/round2__anthropic__claude-sonnet-4-6__batch_manifest__msgbatch_....json")
# round2_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round2_batch_info["data_root"])
# round2_batch_info

In [31]:
round2_output_path = download_anthropic_batch_results(
    batch_id=round2_batch_info["batch_id"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    round_name="round2",
)

round2_output_path

Downloaded/streamed 5,400 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/raw_outputs/round2__msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ__results.jsonl


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/raw_outputs/round2__msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ__results.jsonl')

In [32]:
round2_parse_summary = parse_anthropic_batch_output_to_standard_files(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    parsed_dir=DIRS["round2_parsed"],
    round_name="round2",
    batch_id=round2_batch_info["batch_id"],
)

round2_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])
print(round2_df.shape)
display(round2_df["status"].value_counts(dropna=False))
round2_df.head()

{
  "round": "round2",
  "batch_id": "msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ",
  "n_records": 5400,
  "n_success": 5400,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/parsed/round2__anthropic__claude-sonnet-4-6__msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/parsed/round2__anthropic__claude-sonnet-4-6__msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/02_round2/parsed/round2__anthropic__claude-sonnet-4-6__msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ__parsed.pkl"
}
(5400, 31)


status
success    5400
Name: count, dtype: int64

,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,status,text,provider_response_id,stop_reason,usage,error,batch_custom_id,batch_output_file,batch_id,raw_result_type
0,anthropic,claude-sonnet-4-6,2,slogan_smartphone,slogan,diverge,base,base_069,base_069__a1,1,...,success,Silence the noise. Own the moment.,msg_01VMfvu7Ah8rx2gp6SSFzRsD,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__9f592bde10e6000ef7fde7b8,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,succeeded
1,anthropic,claude-sonnet-4-6,2,aut_shoe,aut,vanilla,triad,triad_023,triad_023__a1,1,...,success,A shoe's stiff sole can be repurposed as a erg...,msg_01Ecna2gTL5fjiz3gkH2KbrS,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__4d15c38c8823420d14cfd32b,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,succeeded
2,anthropic,claude-sonnet-4-6,2,slogan_smartphone,slogan,vanilla,base,base_051,base_051__a1,1,...,success,Capture tomorrow before it slips away.,msg_0156mgVM7qVBGzPhSyuqrNDS,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__a0bc972e6443f25d56d5fe88,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,succeeded
3,anthropic,claude-sonnet-4-6,2,story_jungle,story,vanilla,dyad,dyad_062,dyad_062__a2,2,...,success,Priya had been warned about the jungle twice —...,msg_01LghJoQtPV2EuyrsXFzL9bH,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__535334f02f212d79b4d59e06,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,succeeded
4,anthropic,claude-sonnet-4-6,2,slogan_soda,slogan,vanilla,triad,triad_013,triad_013__a3,3,...,success,Crack it open. Wake everything up.,msg_014AGgxTJSH3CYDJiVzcbqYp,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__625145abb0828409fa52d09f,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,succeeded


In [33]:
expected_round2 = expected_round1
actual_round2 = len(round2_df)

print("Expected Round 2 rows:", expected_round2)
print("Actual Round 2 rows:  ", actual_round2)

display(round2_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round2 != expected_round2:
    print("WARNING: Row count mismatch. Inspect errors before compiling.")

if (round2_df["status"] != "success").any():
    print("WARNING: Some Round 2 calls failed or returned empty text.")
    display(round2_df[round2_df["status"] != "success"].head(20))
else:
    print("Round 2 looks complete.")

Expected Round 2 rows: 5400
Actual Round 2 rows:   5400


,task_id,strategy,condition,status,n
0,aut_button,diverge,base,success,150
1,aut_button,diverge,dyad,success,150
2,aut_button,diverge,triad,success,150
3,aut_button,vanilla,base,success,150
4,aut_button,vanilla,dyad,success,150
5,aut_button,vanilla,triad,success,150
6,aut_shoe,diverge,base,success,150
7,aut_shoe,diverge,dyad,success,150
8,aut_shoe,diverge,triad,success,150
9,aut_shoe,vanilla,base,success,150


Round 2 looks complete.


In [34]:
def normalize_for_analysis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in [
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]:
        if col not in out.columns:
            out[col] = None

    keep_cols = [
        "provider",
        "model",
        "round",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "agent_index",
        "request_key",
        "status",
        "text",
        "temperature",
        "max_output_tokens",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
        "provider_response_id",
        "stop_reason",
        "usage",
        "error",
        "batch_id",
        "batch_custom_id",
        "batch_output_file",
        "parsed_at_utc",
    ]

    existing_keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[existing_keep_cols].copy()

    out["text_clean"] = out["text"].map(clean_model_text)
    out["is_success"] = out["status"].eq("success")

    return out


round1_analysis_df = normalize_for_analysis(round1_df)
round2_analysis_df = normalize_for_analysis(round2_df)

full_long_df = pd.concat([round1_analysis_df, round2_analysis_df], ignore_index=True)

sort_cols = ["task_id", "strategy", "condition", "group_id", "agent_index", "round"]
full_long_df = full_long_df.sort_values(sort_cols).reset_index(drop=True)

print(full_long_df.shape)
display(full_long_df.groupby(["round", "task_id", "strategy", "condition", "status"]).size().reset_index(name="n").head(30))

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
full_csv_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.csv"
full_pkl_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.pkl"

full_long_df.to_csv(full_csv_path, index=False)
full_long_df.to_pickle(full_pkl_path)

print(full_csv_path)
print(full_pkl_path)

(10800, 29)


,round,task_id,strategy,condition,status,n
0,1,aut_button,diverge,base,success,150
1,1,aut_button,diverge,dyad,success,150
2,1,aut_button,diverge,triad,success,150
3,1,aut_button,vanilla,base,success,150
4,1,aut_button,vanilla,dyad,success,150
5,1,aut_button,vanilla,triad,success,150
6,1,aut_shoe,diverge,base,success,150
7,1,aut_shoe,diverge,dyad,success,150
8,1,aut_shoe,diverge,triad,success,150
9,1,aut_shoe,vanilla,base,success,150


ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/03_compiled/anthropic__claude-sonnet-4-6__deflect_creativity__full_long__20260517_213543.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/03_compiled/anthropic__claude-sonnet-4-6__deflect_creativity__full_long__20260517_213543.pkl


In [35]:
r1_small = full_long_df[full_long_df["round"].eq(1)].copy()
r2_small = full_long_df[full_long_df["round"].eq(2)].copy()

merge_keys = ["provider", "model", "task_id", "task_family", "strategy", "condition", "group_id", "agent_id", "agent_index"]

wide_df = r1_small[merge_keys + ["request_key", "status", "text_clean", "batch_id", "usage"]].rename(
    columns={
        "request_key": "round1_request_key",
        "status": "round1_status",
        "text_clean": "round1_text",
        "batch_id": "round1_batch_id",
        "usage": "round1_usage",
    }
).merge(
    r2_small[merge_keys + [
        "request_key",
        "status",
        "text_clean",
        "batch_id",
        "usage",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]].rename(
        columns={
            "request_key": "round2_request_key",
            "status": "round2_status",
            "text_clean": "round2_text",
            "batch_id": "round2_batch_id",
            "usage": "round2_usage",
        }
    ),
    on=merge_keys,
    how="outer",
    validate="one_to_one",
)

wide_df = wide_df.sort_values(["task_id", "strategy", "condition", "group_id", "agent_index"]).reset_index(drop=True)

wide_csv_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.csv"
wide_pkl_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.pkl"

wide_df.to_csv(wide_csv_path, index=False)
wide_df.to_pickle(wide_pkl_path)

print(wide_df.shape)
print(wide_csv_path)
print(wide_pkl_path)
wide_df.head()

(5400, 23)
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/03_compiled/anthropic__claude-sonnet-4-6__deflect_creativity__ego_wide__20260517_213543.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/03_compiled/anthropic__claude-sonnet-4-6__deflect_creativity__ego_wide__20260517_213543.pkl


,provider,model,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,round1_request_key,...,round1_usage,round2_request_key,round2_status,round2_text,round2_batch_id,round2_usage,self_round1_request_key,peer_round1_agent_ids,self_round1_text,peer_round1_texts_json
0,anthropic,claude-sonnet-4-6,aut_button,aut,diverge,base,base_001,base_001__a1,1,r1__4718bbc27c8658f239ac6345,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__a876b11238b185a7d5676f33,success,Using a button's raised rim as a miniature mol...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__4718bbc27c8658f239ac6345,NaN,Using a button as a tiny canvas for miniature ...,[]
1,anthropic,claude-sonnet-4-6,aut_button,aut,diverge,base,base_002,base_002__a1,1,r1__717737e987a9810b894d47bb,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__0c51631c832e7ea01f553e0d,success,Pressing a button into soft clay or wax to cre...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__717737e987a9810b894d47bb,NaN,Using a button as a tiny palette for mixing sm...,[]
2,anthropic,claude-sonnet-4-6,aut_button,aut,diverge,base,base_003,base_003__a1,1,r1__a6575db26dffd03b41b45bd1,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__6c58966bd746dab0f0a5896f,success,Use a button's raised rim and central holes as...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__a6575db26dffd03b41b45bd1,NaN,Glue a collection of mismatched buttons onto a...,[]
3,anthropic,claude-sonnet-4-6,aut_button,aut,diverge,base,base_004,base_004__a1,1,r1__58b3a313f56345a7f041fa17,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__700576ba9d04fa4b88cf3432,success,Place a button at the base of a potted plant's...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__58b3a313f56345a7f041fa17,NaN,"Use a button as a tiny painter's palette, plac...",[]
4,anthropic,claude-sonnet-4-6,aut_button,aut,diverge,base,base_005,base_005__a1,1,r1__3d611088b91f87e5302886de,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__8be51dc1d8c0a7474f675ed7,success,Press a button repeatedly into soft clay or wa...,msgbatch_01QLKjYYpyz9UDpxFu5mNqUJ,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__3d611088b91f87e5302886de,NaN,Use a large decorative button as a miniature p...,[]


In [36]:
def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\w'-]+\b", text))


def sentence_count_rough(text: str) -> int:
    if not isinstance(text, str):
        return 0
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)


validation_df = full_long_df.copy()
validation_df["word_count"] = validation_df["text_clean"].map(word_count)
validation_df["rough_sentence_count"] = validation_df["text_clean"].map(sentence_count_rough)

slogan_violations = validation_df[
    validation_df["task_family"].eq("slogan")
    & validation_df["is_success"]
    & validation_df["word_count"].gt(6)
].copy()

story_sentence_violations = validation_df[
    validation_df["task_family"].eq("story")
    & validation_df["is_success"]
    & validation_df["rough_sentence_count"].ne(8)
].copy()

print("Slogan >6-word violations:", len(slogan_violations))
display(slogan_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "word_count"]].head(20))

print("Story rough sentence-count violations:", len(story_sentence_violations))
display(story_sentence_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "rough_sentence_count"]].head(20))

validation_csv_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__validation_flags__{timestamp}.csv"
validation_df.to_csv(validation_csv_path, index=False)

validation_csv_path

Slogan >6-word violations: 43


,round,task_id,strategy,condition,agent_id,text_clean,word_count
3710,1,slogan_smartphone,diverge,base,base_056__a1,Outthink tomorrow. It starts in your hand.,7
3919,2,slogan_smartphone,diverge,dyad,dyad_005__a2,Silence the gap between dreams and reality.,7
4157,2,slogan_smartphone,diverge,dyad,dyad_065__a1,Silence the gap between dreams and doing.,7
4163,2,slogan_smartphone,diverge,dyad,dyad_066__a2,Silence the gap between dreams and done.,7
4303,2,slogan_smartphone,diverge,triad,triad_018__a1,Silence the gap between you and everything.,7
4390,1,slogan_smartphone,diverge,triad,triad_032__a3,Outsmart tomorrow. Today fits in your pocket.,7
4465,2,slogan_smartphone,diverge,triad,triad_045__a1,Silence the gap between you and everything.,7
4539,2,slogan_smartphone,vanilla,base,base_020__a1,Redefine possible. One pocket at a time.,7
4565,2,slogan_smartphone,vanilla,base,base_033__a1,Blur the line between you and extraordinary.,7
4571,2,slogan_smartphone,vanilla,base,base_036__a1,Redefine possible. One pocket at a time.,7


Story rough sentence-count violations: 910


,round,task_id,strategy,condition,agent_id,text_clean,rough_sentence_count
7207,2,story_jungle,diverge,base,base_004__a1,Priya had been following the bird for forty mi...,7
7208,1,story_jungle,diverge,base,base_005__a1,The map had been folded and unfolded so many t...,7
7211,2,story_jungle,diverge,base,base_006__a1,Dayo had been warned not to follow the river p...,7
7215,2,story_jungle,diverge,base,base_008__a1,Dario had told no one he was sneaking away fro...,7
7219,2,story_jungle,diverge,base,base_010__a1,Tomás had been warned about the jungle—by his ...,7
7227,2,story_jungle,diverge,base,base_014__a1,Tomás had only agreed to the jungle trek becau...,7
7238,1,story_jungle,diverge,base,base_020__a1,The map my grandmother left me was drawn in in...,7
7243,2,story_jungle,diverge,base,base_022__a1,Dario found the temple on the third day of bei...,7
7245,2,story_jungle,diverge,base,base_023__a1,"The drone had been Marcus's idea, but it was M...",9
7246,1,story_jungle,diverge,base,base_024__a1,"The compass had been lying for three days, and...",7


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/03_compiled/anthropic__claude-sonnet-4-6__deflect_creativity__validation_flags__20260517_213543.csv')

In [37]:
final_manifest = {
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "round1_batch_info": round1_batch_info,
    "round2_batch_info": round2_batch_info,
    "round1_parse_summary": round1_parse_summary,
    "round2_parse_summary": round2_parse_summary,
    "compiled_long_csv": str(full_csv_path),
    "compiled_long_pkl": str(full_pkl_path),
    "compiled_wide_csv": str(wide_csv_path),
    "compiled_wide_pkl": str(wide_pkl_path),
    "validation_csv": str(validation_csv_path),
    "completed_at_utc": now_iso(),
}

final_manifest_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__final_manifest__{timestamp}.json"
write_json(final_manifest_path, final_manifest)

final_manifest_path

PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/run_20260517_205619__040c8ea3/03_compiled/anthropic__claude-sonnet-4-6__deflect_creativity__final_manifest__20260517_213543.json')